### 복습 (데이터프레임)
1. 민원(콜센터) 질의응답_다산콜센터_일반행정 문의_Training.json 파일을 로드
2. 고객질문(요청), 상담사답변 컬럼의 문자 정규화 (특문 제거, 공백 제거)
3. 고객질문에 대한 즉각적인 상담ㄷ사의 답변이 있는 행들만 남기고 나머지는 제거
4. 고객의 질문과 상담사의 답변을 하나의 행으로 결합
5. 인덱스를 초기화하고 label 컬럼을 생성하여 1을 대입


In [245]:
import pandas as pd
import re
import numpy as np

In [246]:
data = pd.read_json("../data/민원(콜센터) 질의응답_다산콜센터_일반행정 문의_Training.json")
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50336 entries, 0 to 50335
Data columns (total 15 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   도메인        50336 non-null  object
 1   카테고리       50336 non-null  object
 2   대화셋일련번호    50336 non-null  object
 3   화자         50336 non-null  object
 4   문장번호       50336 non-null  int64 
 5   고객의도       50336 non-null  object
 6   상담사의도      50336 non-null  object
 7   QA         50336 non-null  object
 8   고객질문(요청)   50336 non-null  object
 9   상담사질문(요청)  50336 non-null  object
 10  고객답변       50336 non-null  object
 11  상담사답변      50336 non-null  object
 12  개체명        50336 non-null  object
 13  용어사전       50336 non-null  object
 14  지식베이스      50336 non-null  object
dtypes: int64(1), object(14)
memory usage: 5.8+ MB


In [247]:
data.head(10)

,도메인,카테고리,대화셋일련번호,화자,문장번호,고객의도,상담사의도,QA,고객질문(요청),상담사질문(요청),고객답변,상담사답변,개체명,용어사전,지식베이스
0,다산콜센터,일반행정 문의,B2240,고객,1,지방세납부,,Q,지방세를 내려면 어떻게 해야됩니까?,,,,지방세,지방세/세금,"지방세,세금"
1,다산콜센터,일반행정 문의,B2240,상담사,2,,지방세납부,A,,,,이용하시는 은행의 사이트에서 지방세 납부가 가능합니다.,"은행, 사이트, 지방세, 납부",은행/공공기관/ 지방세/세금,"사이트,세금"
2,다산콜센터,일반행정 문의,B2240,고객,3,지방세납부,,Q,은행 어플에서도 됩니까?,,,,"은행, 어플",은행/공공기관,"어플,공공기관"
3,다산콜센터,일반행정 문의,B2240,상담사,4,,지방세납부,Q,,어떤 은행을 이용하고 계십니까?,,,은행,은행/공공기관,"은행,공공기관"
4,다산콜센터,일반행정 문의,B2240,고객,5,지방세납부,,A,,,기업은행을 이용하고 있습니다.,,기업은행,기업은행/상호,"기업은행,상호"
5,다산콜센터,일반행정 문의,B2240,상담사,6,,지방세납부,A,,,,그럼 스마트폰에서 기업은행 어플을 설치하시면 납부가 가능합니다.,"스마트폰, 기업은행, 어플, 납부",스마트폰/전자기기/ 기업은행/상호,"기업은행,상호"
6,다산콜센터,일반행정 문의,B2240,고객,7,지방세납부,,Q,은행을 직접방문해도 됩니까?,,,,은행,은행/공공기관,"은행,공공기관"
7,다산콜센터,일반행정 문의,B2240,상담사,8,,지방세납부,A,,,,방문납부도 가능합니다.,,,
8,다산콜센터,일반행정 문의,B2240,고객,9,지방세납부,,Q,은행위치 좀 알 수 있습니까?,,,,은행,은행/공공기관,"은행,공공기관"
9,다산콜센터,일반행정 문의,B2240,상담사,10,,지방세납부,Q,,어느지점으로 안내해드릴까요?,,,,,


In [248]:
# 텍스트 정규화 함수
def normalize_token_text(text:str) -> str:
    text= re.sub(r'[^가-힣a-zA-Z0-9\s\.]', " ", text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

data[['고객질문(요청)', '상담사답변']] = data[['고객질문(요청)', '상담사답변']].applymap(normalize_token_text)

C:\Users\abohv\AppData\Local\Temp\ipykernel_13188\3671242311.py:7: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  data[['고객질문(요청)', '상담사답변']] = data[['고객질문(요청)', '상담사답변']].applymap(normalize_token_text)


In [249]:
# 고객질문(요청)컬럼의 이름을 고객질문으로 변경
data.rename(columns={
    '고객질문(요청)' : '고객질문'
}, inplace=True)

In [250]:
flag1 = (data['고객질문'] != '') & \
    (data['상담사답변'].shift(-1) != '')
    
flag2 = (data['상담사답변'] != '') & \
        (data['고객질문'].shift(1) != ''  )

data2 = data.loc[flag1 | flag2,]
data2['상담사답변'] = data2['상담사답변'].shift(-1)
data2 = data2.loc[
    data2['고객질문'] != ''
]
data2 = data2.drop_duplicates('고객질문').reset_index(drop=True)
data2

C:\Users\abohv\AppData\Local\Temp\ipykernel_13188\3143818816.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data2['상담사답변'] = data2['상담사답변'].shift(-1)


,도메인,카테고리,대화셋일련번호,화자,문장번호,고객의도,상담사의도,QA,고객질문,상담사질문(요청),고객답변,상담사답변,개체명,용어사전,지식베이스
0,다산콜센터,일반행정 문의,B2240,고객,1,지방세납부,,Q,지방세를 내려면 어떻게 해야됩니까,,,이용하시는 은행의 사이트에서 지방세 납부가 가능합니다.,지방세,지방세/세금,"지방세,세금"
1,다산콜센터,일반행정 문의,B2240,고객,7,지방세납부,,Q,은행을 직접방문해도 됩니까,,,방문납부도 가능합니다.,은행,은행/공공기관,"은행,공공기관"
2,다산콜센터,일반행정 문의,B2240,고객,13,지방세납부,,Q,버스로 가는 방법도 있습니까,,,도보로 가시는게 빠를 것 같습니다.,버스,버스/교통수단,"버스,교통수단"
3,다산콜센터,일반행정 문의,B2240,고객,15,지방세납부,,Q,다른 납부방법도 있습니까,,,위택스 사이트에서 납부하실수 있습니다.,,납부방법/지방세,
4,다산콜센터,일반행정 문의,B2240,고객,17,지방세납부,,Q,사이트 주소가 어떻게 됩니까,,,www.wetax.go.kr 입니다.,사이트,납부방법/지방세/ 위텍스/ 납부,"사이트,납부"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12385,다산콜센터,일반행정 문의,B35809,고객,9,여성전용아파트,,Q,꼭 서울시에 근무해야하나요,,,서울소재 직장근무로 제한하고있습니다.,"서울시, 근무","서울시/서울/서울특별시, 근무",근무
12386,다산콜센터,일반행정 문의,B35809,고객,11,여성전용아파트,,Q,임대료는 어떻게 되나요,,,62400원 입니다.,임대료,"임대료, 월세","임대료,월세"
12387,다산콜센터,일반행정 문의,B35809,고객,13,여성전용아파트,,Q,보증금도 있나요,,,1423200원 입니다.,보증금,보증금/예치금,"보증금,예치금"
12388,다산콜센터,일반행정 문의,B35809,고객,17,여성전용아파트,,Q,입주순위도 있나요,,,1 3순위가 있습니다.,"입주, 순위","입주/이사, 순위/순서","순위,순서"


In [251]:
QnA = data2[['고객질문', '상담사답변']]

df = pd.DataFrame(QnA)

df.reset_index(drop=True, inplace=True)


In [252]:
df['label'] = 1

In [253]:
df.head(10)

,고객질문,상담사답변,label
0,지방세를 내려면 어떻게 해야됩니까,이용하시는 은행의 사이트에서 지방세 납부가 가능합니다.,1
1,은행을 직접방문해도 됩니까,방문납부도 가능합니다.,1
2,버스로 가는 방법도 있습니까,도보로 가시는게 빠를 것 같습니다.,1
3,다른 납부방법도 있습니까,위택스 사이트에서 납부하실수 있습니다.,1
4,사이트 주소가 어떻게 됩니까,www.wetax.go.kr 입니다.,1
5,다른곳에서는 납부할수 없습니까,이용하시는 은행의 사이트에서도 지방세 납부가 가능합니다.,1
6,지방세는 조회할 수 있습니까,간단한 본인확인 후 안내해드리겠습니다.,1
7,어떻게 납부합니까,온라인으로는 위택스나 은행 홈페이지에서 가능합니다.,1
8,서울시주최 페스티벌 예매해놨는데 예정대로 진행됩니까,현재로썬 진행될 예정입니다.,1
9,코로나로 다른 축제들은 취소됐는데 이건 취소 안됩니까,네 철저한 방역수칙하에 진행할 예정입니다.,1


In [254]:
q_df = df.copy()

In [261]:
len(q_df)

12390

In [255]:
# label -> 1을 채운 이유 -> 해당하는 행의 질문과 답변은 정상적인 답변입니다.
# label -> 0인 데이터들을 생성 -> 기존의 질문과 답변에서 질문은 그대로 유지한채 답변은 원래의 답변을 제외한 다른 랜덤한 답으로 채운다.

# 또는 질문+답변 컬럼만 리스트로 만들려면
answer_list = q_df[['고객질문', '상담사답변']].values.tolist()


In [ ]:
# 질문은 유지한채 답변을 다른 답변으로 변경하여 새로운 2차원 리스트 생성
neg_list = []
for q, a in answer_list:
    # q -> 질문
    # a -> 답변
    cand = [a for q2, a2 in answer_list if a2 != a]
    # 틀린 답변 하나를 선택
    neg_a = np.random.choice(cand)
    neg_list.append([q, neg_a])

In [257]:
neg_df = pd.DataFrame(neg_list, columns=['고객질문', '상담사답변'])
neg_df.head()

,고객질문,상담사답변
0,지방세를 내려면 어떻게 해야됩니까,이용하시는 은행의 사이트에서 지방세 납부가 가능합니다.
1,은행을 직접방문해도 됩니까,방문납부도 가능합니다.
2,버스로 가는 방법도 있습니까,도보로 가시는게 빠를 것 같습니다.
3,다른 납부방법도 있습니까,위택스 사이트에서 납부하실수 있습니다.
4,사이트 주소가 어떻게 됩니까,www.wetax.go.kr 입니다.


In [258]:
neg_df['label'] = 0

In [262]:
# q_df와 neg_df를 단순한 행 결합
dataset_df = pd.concat([q_df.head(1000), neg_df.head(1000)], axis=0,ignore_index=True)

In [263]:
dataset_df['label'].value_counts()

label
1    1000
0    1000
Name: count, dtype: int64

In [265]:
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split

In [284]:
# train, test 셋으로 데이터프레임을 분할
train_df, test_df = train_test_split(
    dataset_df, test_size=0.2, random_state=42, stratify=dataset_df['label']
)
train_ds = Dataset.from_pandas(train_df.reset_index(drop=True))
test_ds= Dataset.from_pandas(test_df.reset_index(drop=True))
ds = DatasetDict(
    {
        'train' : train_ds,
        'validation' : test_ds
    }
)
ds

DatasetDict({
    train: Dataset({
        features: ['고객질문', '상담사답변', 'label'],
        num_rows: 1600
    })
    validation: Dataset({
        features: ['고객질문', '상담사답변', 'label'],
        num_rows: 400
    })
})

In [285]:
import torch
from transformers import AutoTokenizer, DataCollatorWithPadding
# BertForSequenceClassification : BertModel -> <CLS> 토큰 vector 추출 -> dropout -> 
from transformers import Trainer, TrainingArguments, BertForSequenceClassification
from sklearn.metrics import accuracy_score, f1_score

In [286]:
# 토큰화
MODEL_NAME = 'skt/kobert-base-v1'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)
max_len = 256       # 장문의 데이터인 경우에는 512까지 조절

def tok_fn(batch):
    # token화 할 데이터가 2개의 컬럼에 나눠져있다.
    enc = tokenizer(
        batch['고객질문'],
        batch['상담사답변'],
        truncation = True,
        max_length = max_len
    )
    
    # 일반적인 kobert 모델에서는 token_type_ids의 데이터 영역 필요X
    enc.pop("token_type_ids", None)     # 완전히 제거
    return enc

# remove_columns -> 특정 컬럼의 데이터는 제외 -> label컬럼을 제외한 나머지 모두
tok_ds =ds.map(tok_fn, batched=True, remove_columns=[col for col in dataset_df.columns if col not in ['label']],
    # 캐시 사용 안함
    load_from_cache_file = False,
    desc = 'tokeinze clean'
    )

tokeinze clean: 100%|██████████| 400/400 [00:00<00:00, 7665.13 examples/s]


In [287]:
tok_ds

DatasetDict({
    train: Dataset({
        features: ['label', 'input_ids', 'attention_mask'],
        num_rows: 1600
    })
    validation: Dataset({
        features: ['label', 'input_ids', 'attention_mask'],
        num_rows: 400
    })
})

In [288]:
# 배치마다 동적으로 padding토큰을 추가
collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

In [289]:
# BertModel -> CLS -> dropout -> linear
model = BertForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at skt/kobert-base-v1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [298]:
# 평가 지표 함수
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis= -1)
    return {
        'accuracy_score' : accuracy_score(labels, preds),
        'f1' : f1_score(labels, preds)
    }

In [299]:
args = TrainingArguments(
    output_dir = '/kobert_pair_cls',
    eval_strategy= 'epoch',
    save_strategy='epoch',
    logging_steps=50,
    learning_rate = 5e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,
    report_to=[]
)

In [300]:
trainer = Trainer(
    model = model,
    args = args,
    train_dataset= tok_ds['train'],
    eval_dataset= tok_ds['validation'],
    tokenizer=tokenizer,
    data_collator= collator,
    compute_metrics= compute_metrics
    
)

C:\Users\abohv\AppData\Local\Temp\ipykernel_13188\1111433693.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [301]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy Score,F1
1,0.697700,0.693891,0.500000,0.000000
2,0.699500,0.693336,0.500000,0.666667
3,0.697900,0.693199,0.500000,0.666667


c:\Users\abohv\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
c:\Users\abohv\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


TrainOutput(global_step=600, training_loss=0.6997638320922852, metrics={'train_runtime': 1029.9758, 'train_samples_per_second': 4.66, 'train_steps_per_second': 0.583, 'total_flos': 94892646559680.0, 'train_loss': 0.6997638320922852, 'epoch': 3.0})

In [309]:
# 평가
def score_pairs(question, answers):
    # 질문 1개 -> 답변 여러개
    questions = [question] * len(answers)
    enc = tokenizer(
        questions,
        answers,
        return_tensors = 'pt',
        truncation = True,
        max_length = max_len,
        padding = True
    )
    
    # token_type_ids 제거
    enc.pop('token_type_ids', None)
    
    
    # 인코딩 데이터를 딕셔너리 형태로 변환
    inputs = {
        k : v for k, v in enc.items()
    }
    model.eval()
    with torch.no_grad():
        logits = model(**inputs).logits
        probs = torch.softmax(logits, dim= -1)[:, 1]    # label 1의 확률
            # probs 클수록 올바른 답변
    return probs.cpu().tolist()

In [310]:
question =  '지방세 납부는 어디서 할 수 있을까요?'
answers = [
    '이용하시는 은행 사이트나 앱에서 지방세 납부가 가능합니다',
    '동물 등록은 거주지 구청에서 처리하셔야 합니다',
    '출입구 관련 업무는 외교부에서 담당합니다'
]


scores = score_pairs(question, answers)

In [312]:
scores

[0.5097306370735168, 0.5097324252128601, 0.5097281336784363]

In [317]:
best_idx = int(np.argmax(scores))
for i, (a, s) in enumerate(zip(answers, scores)):
    # 1 -> index
    # a - > 답변
    # s -> 1의 적합 확률
    print(f"{i}  {a}  ->  적합확률  : {round(s, 3)}")
print(f"적합한 답변은 {best_idx} : {answers[best_idx]}")

0  이용하시는 은행 사이트나 앱에서 지방세 납부가 가능합니다  ->  적합확률  : 0.51
1  동물 등록은 거주지 구청에서 처리하셔야 합니다  ->  적합확률  : 0.51
2  출입구 관련 업무는 외교부에서 담당합니다  ->  적합확률  : 0.51
적합한 답변은 1 : 동물 등록은 거주지 구청에서 처리하셔야 합니다
